# Library Management System — C++ Project Notes

**Course:** Codecademy — Learn Intermediate C++ (Storage Classes)

**Project goal:** Build a `Library` class that manages books, borrows, reservations, and a budget — while practicing **static members**, **global variables (`extern`)**, **`const` member functions**, and **`mutable` members**.

**Files:** `Library.hpp` (class declaration) · `Library.cpp` (implementations) · `main.cpp` (driver)

## Tasks 1–7: Class setup, `addBook`, `borrowBook`, `listBooks`

- Class declaration goes in the **`.hpp`**, function implementations in the **`.cpp`** using `Library::functionName` syntax.
- `addBook` only adds while `bookList.size() < 100`; otherwise prints `Library is full!`
- `borrowBook` searches with a loop + `found` flag, calls `removeElement(bookList, title)` on success, prints `Book borrowed!` / `Book not found!`, returns `true`/`false`.

**Style notes from grading:**
- Loop variable naming: each element of `bookList` is a single `title`, not `titles`.
- `for (auto book : bookList)` **copies** each string — `for (const auto& book : bookList)` avoids the copy.
- `std::find` from `<algorithm>` can replace the manual search loop + flag entirely.
- **Header guards:** `Library.hpp` should start with `#pragma once` (or an `#ifndef` guard) so double-inclusion doesn't break the build.

## Tasks 8–11: `static` members — counting across ALL instances

- A **`static` data member** belongs to the *class*, not to any one object — every `Library` instance shares the same `totalBorrowedBooks`.
- Declared inside the class, but **defined/initialized at file scope in the `.cpp`**:

```cpp
// Library.hpp (inside class)
private:
  static int totalBorrowedBooks;
public:
  static int getTotalBorrowedBooks();

// Library.cpp (at file scope)
int Library::totalBorrowedBooks = 0;
```

- The increment lives in `borrowBook`, in the **found branch only** (a failed borrow isn't a borrow), and **before** `return true;` — code after a `return` in the same block is unreachable and never runs.

```cpp
if (found) {
  removeElement(bookList, title);
  totalBorrowedBooks++;
  cout << "Book borrowed!" << endl;
  return true;
}
```

## Tasks 12–13: Global variables and `extern`

- `extern double libraryBudget;` is a **declaration** — a promise that the variable is *defined* somewhere else. No memory is allocated by an `extern` line.
- The `extern` declaration belongs in the **header** (`Library.hpp`) so every file that includes it knows the variable exists. (First attempt put it in `Library.cpp` — then `main.cpp` would never learn about it.)
- Exactly **one** file holds the real definition (no `extern`):

```cpp
// Library.hpp
extern double libraryBudget;

// main.cpp
double libraryBudget = 5000.0;
```

- ⚠️ The course itself warns: globals like this are for practice — real projects use better techniques (pass parameters, member variables, etc.).
- **Lesson learned the hard way:** Codecademy's output checker is *exact-match* — "budgert" ≠ "budget". Read print statements character by character.

## Tasks 14–15: Changing a function signature + budget deduction

- When a function's parameter list changes in the header, the `.cpp` definition **must match exactly** — declaration and definition are checked against each other at compile time.
- Watch parameter-name typos: the task wanted `bookCost`, first attempt typed `bookConst` (compiles, but fails the checker and confuses readers).

**Operator confusion worked through in this session:**

| Attempt | What it actually does |
|---|---|
| `--bookCost;` | subtracts exactly **1** from the *price* — wrong variable, wrong amount |
| `bookCost -= bookCost;` | price minus itself = sets the price to **0**, budget untouched |
| `libraryBudget -= bookCost;` | ✅ budget shrinks by the cost — correct |

- General shape: `<thing that should shrink> -= <amount to shrink it by>;`
- Placement matters: the deduction goes **inside** the `if (bookList.size() < 100)` branch, next to the `push_back`, so cost is only incurred when the book is actually added:

```cpp
void Library::addBook(string title, double bookCost) {
  if (bookList.size() < 100) {
    bookList.push_back(title);
    libraryBudget -= bookCost;
  } else {
    cout << "Library is full!" << endl;
  }
}
```

- Note: the task did **not** ask to check the budget is sufficient — adding a 6000-cost book would drive the budget negative. (Noticed, not fixed.)

## Tasks 16–17: `const` member functions and `mutable`

- `void reserveBook(string title) const;` — the `const` **after the parameter list** makes this a *constant member function*: it promises not to modify any member variables of the object.
- But reserving a book needs to *record* something → collision. The escape hatch is **`mutable`**: a `mutable` member may be modified even inside a `const` function.

```cpp
private:
  vector<string> bookList;
  static int totalBorrowedBooks;
  mutable int reservedCount;
  mutable vector<string> reservedBooks;
```

- Mental test: remove `mutable` from either one and the compiler rejects the `push_back` / `++` inside `reserveBook`.
- This is the core concept of the project: `const` = the contract, `mutable` = the deliberate, documented exception.

## Task 18: The constructor — initializing members

- Built-in types like `int` are **not** auto-initialized — an unconstructed `reservedCount` holds *garbage* (whatever was in that memory). Incrementing garbage gives garbage.
- The constructor declared back in Task 2 was never defined — that's a **linker error** waiting to fire the moment `main()` creates a `Library`. Defining it fixed the dangling thread:

```cpp
// straightforward version
Library::Library() {
  reservedCount = 0;
}

// idiomatic version — member initializer list
Library::Library() : reservedCount(0) {}
```

- Both work identically here. The initializer list (after the `:`) initializes the member *as it's constructed*, and it's the **only** way to initialize `const` members and references — worth building the habit.

## Task 19: `reserveBook` implementation

- Requirements: add to `reservedBooks` **if and only if** the title exists in `bookList` AND is not already in `reservedBooks`. On success: increment `reservedCount`, print `Book reserved: title` and `Reserved book count: count`. On failure: print the task's exact failure string.

```cpp
void Library::reserveBook(string title) const {
  bool inLibrary = false;
  for (auto book : bookList) {
    if (book == title) { inLibrary = true; break; }
  }

  bool alreadyReserved = false;
  for (auto book : reservedBooks) {
    if (book == title) { alreadyReserved = true; break; }
  }

  if (inLibrary && !alreadyReserved) {
    reservedBooks.push_back(title);
    reservedCount++;
    cout << "Book reserved: " << title << endl;
    cout << "Reserved book count: " << reservedCount << endl;
  } else {
    cout << "<exact failure string from the task>" << endl;
  }
}
```

- Two searches → two booleans → success requires `inLibrary && !alreadyReserved` (`!` = "not").
- Order in the success branch: add → increment → print, so the printed count includes the book just reserved.
- The whole function only compiles *because* `reservedBooks` and `reservedCount` are `mutable`.

## Tasks 20–21: Using the class in `main()`

```cpp
int main() {
  cout << "Current library budget is: " << libraryBudget << endl;

  Library myLibrary;   // constructor runs here -> reservedCount = 0

  myLibrary.addBook("The Great Gatsby", 300);
  myLibrary.addBook("1984", 400);
  myLibrary.addBook("To Kill a Mockingbird", 350);
}
```

- Pattern: `object.function(arguments)`.
- After the three `addBook` calls the budget has silently dropped 5000 → **3950** — expect that number if a later task prints the budget again.
- ⚠️ Copy/paste hazard: task descriptions use curly “smart quotes” — code needs straight quotes `"` or it won't compile.

## Key concepts cheat sheet (storage classes)

| Keyword / idea | What it does | Where it appeared |
|---|---|---|
| `static` data member | One shared variable for **all** instances of the class; defined at file scope in the `.cpp` | `totalBorrowedBooks` |
| `static` member function | Callable on the class itself; can only touch static members | `getTotalBorrowedBooks()` |
| `extern` | *Declares* a global defined in another file; goes in the header, definition in exactly one `.cpp` | `libraryBudget` |
| `const` member function | `const` after the parameter list: promises not to modify the object | `reserveBook()` |
| `mutable` | Member that **may** be modified even in a `const` function | `reservedCount`, `reservedBooks` |
| Constructor | Runs at object creation; initialize built-in types or they hold garbage | `Library() : reservedCount(0) {}` |
| `x -= y` | `x = x - y` (vs. `--x` which subtracts exactly 1) | budget deduction |

**Recurring gotchas from this session:**
- Declaration (`.hpp`) and definition (`.cpp`) signatures must match exactly.
- Declared-but-never-defined functions compile fine and explode at **link** time, the first time they're used.
- Exact-match output checkers: typos like "budgert" fail silently-looking code.
- Code after `return` never runs.